# 02 — Inspect Pipeline Results (Post-Pipeline)

Inspect what the CI/CD pipeline produced after at least one successful run.

**Prerequisite:** at least one green run of the GitHub Actions pipeline.

**Sections:**
1. Inspect agent versions on the shelf
2. Visualize semantic view eval scores
3. Visualize agent eval scores
4. Compare scores across pipeline runs
5. Chat with the promoted default version

In [ ]:
import snowflake.snowpark.context as ctx
import pandas as pd
import matplotlib.pyplot as plt

session = ctx.get_active_session()

## 1 — Agent Versions on the Shelf

In [ ]:
versions = session.sql('SHOW VERSIONS IN AGENT SV_EVAL_CICD.APP.GROWTH_AGENT').to_pandas()
print(f'{len(versions)} version(s) on the shelf')
print(versions[['name', 'is_default', 'aliases', 'created_on']].to_string(index=False))

## 2 — Semantic View Eval Scores

Set `run_name` to the run name printed in the `eval_sv` job logs.

In [ ]:
# Replace with the run_name from the eval_sv job logs
sv_run_name = 'REPLACE_WITH_RUN_NAME_FROM_LOGS'

sv_scores = session.sql(f"""
    SELECT metric_name, ROUND(AVG(eval_agg_score), 3) AS avg_score
    FROM TABLE(SNOWFLAKE.LOCAL.GET_ANALYST_AI_EVALUATION_DATA(
        'SV_EVAL_CICD', 'APP', 'GROWTH_ANALYTICS_SV', 'CORTEX ANALYST', '{sv_run_name}'
    ))
    GROUP BY 1
""").to_pandas()
print(sv_scores.to_string(index=False))

In [ ]:
thresholds = {'sql_correctness': 0.35}

fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(sv_scores['METRIC_NAME'], sv_scores['AVG_SCORE'], color='steelblue')
for metric, thr in thresholds.items():
    ax.axhline(y=thr, color='red', linestyle='--', alpha=0.6, label=f'{metric} threshold ({thr})')
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title('Semantic View Eval Scores')
ax.legend()
plt.tight_layout()
plt.show()

## 3 — Agent Eval Scores

Set `agent_run_name` to the run name printed in the `eval` job logs.

In [ ]:
# Replace with the run_name from the eval job logs
agent_run_name = 'REPLACE_WITH_RUN_NAME_FROM_LOGS'

agent_scores = session.sql(f"""
    SELECT metric_name, ROUND(AVG(eval_agg_score), 3) AS avg_score
    FROM TABLE(SNOWFLAKE.LOCAL.GET_AI_EVALUATION_DATA(
        'SV_EVAL_CICD', 'APP', 'GROWTH_AGENT', 'CORTEX AGENT', '{agent_run_name}'
    ))
    GROUP BY 1
""").to_pandas()
print(agent_scores.to_string(index=False))

In [ ]:
thresholds = {'answer_correctness': 0.70, 'logical_consistency': 0.70, 'tool_selection_accuracy': 0.70}

fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(agent_scores['METRIC_NAME'], agent_scores['AVG_SCORE'], color='steelblue')
for metric, thr in thresholds.items():
    if metric in agent_scores['METRIC_NAME'].values:
        ax.axhline(y=thr, color='red', linestyle='--', alpha=0.6)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title('Agent Eval Scores (red dashed = promotion threshold 0.70)')
plt.tight_layout()
plt.show()

## 4 — Compare Scores Across Pipeline Runs

Populate `runs` with the `run_name` values from multiple pipeline runs (check the `eval` job logs).

In [ ]:
runs = [
    'RUN_NAME_1',
    'RUN_NAME_2',
    # add more run names here
]

records = []
for run in runs:
    rows = session.sql(f"""
        SELECT '{run}' AS run, metric_name, ROUND(AVG(eval_agg_score), 3) AS avg_score
        FROM TABLE(SNOWFLAKE.LOCAL.GET_AI_EVALUATION_DATA(
            'SV_EVAL_CICD', 'APP', 'GROWTH_AGENT', 'CORTEX AGENT', '{run}'
        ))
        GROUP BY 1, 2
    """).to_pandas()
    records.append(rows)

if records:
    df_all = pd.concat(records)
    pivot = df_all.pivot(index='RUN', columns='METRIC_NAME', values='AVG_SCORE')
    print(pivot.to_string())
else:
    print('Add at least two run names to runs[] above.')

## 5 — Chat with the Promoted Version

The `promote` job sets `DEFAULT_VERSION = LAST`. Chat with the version the pipeline just promoted.

In [ ]:
import json

def ask(question):
    payload = json.dumps({'messages': [{'role': 'user', 'content': [{'type': 'text', 'text': question}]}]})
    row = session.sql(f"""
        SELECT SNOWFLAKE.CORTEX.DATA_AGENT_RUN(
            'SV_EVAL_CICD.APP.GROWTH_AGENT!production',
            $${payload}$$
        ) AS response
    """).collect()[0]
    resp = json.loads(row['RESPONSE'])
    msgs = resp.get('messages', [])
    text = msgs[-1].get('content', [{}])[0].get('text', str(resp)) if msgs else str(resp)
    print(f'Q: {question}')
    print(f'A: {text[:400]}')
    print()

ask('How many users signed up in Q1 2025?')
ask('What was total MRR from paid conversions in January 2025?')
ask('Which channel had the highest ROAS in Q1 2025?')